## Code Completion

<a target="_blank" href="https://colab.research.google.com/github/microsoft/LLMLingua/blob/main/examples/Code.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

In addition to natural language, we also evaluate the capabilities of our method on code. For example, in [**repobench-p**](https://arxiv.org/abs/2306.03091), the original prompts can be particularly long, up to **20k** characters. LongLLMLingua can achieve a 1.4 point improvement at a **6x** compression ratio.

### RepoBench-p

Next, we will demonstrate the use of LongLLMLingua on the **RepoBench-P** dataset (Code Completion Task). This dataset contains samples of incomplete code snippets and several related reference code files. The task is to supplement and complete the next line of code.  
   
The original dataset can be found at https://huggingface.co/datasets/THUDM/LongBench.

In [ ]:
# Install dependency.
!pip install llmlingua datasets

In [ ]:
# Download the original prompt and dataset
from datasets import load_dataset

dataset = load_dataset("THUDM/LongBench", "repobench-p", split="test")

In [ ]:
# Using the OAI
import openai

openai.api_key = "<insert_openai_key>"

In [ ]:
# or Using the AOAI
import openai

openai.api_key = "<insert_openai_key>"
openai.api_base = "https://xxxx.openai.azure.com/"
openai.api_type = "azure"
openai.api_version = "2023-05-15"

### Setup Data

In [ ]:
# select an example from MeetingBank
contexts, question, answer = [
    dataset[1][key] for key in ["context", "input", "answers"]
]
instruction = "Please complete the code given below."
question = question + "\n\nNext line of code:\n"

In [ ]:
# Ground-truth Answer
answer

### The response of Original prompt

In [ ]:
# The response from original prompt, using GPT-4-32k
import json

prompt = "\n\n".join([instruction, contexts, question])

message = [
    {"role": "user", "content": prompt},
]

request_data = {
    "messages": message,
    "max_tokens": 100,
    "temperature": 0,
    "top_p": 1,
    "n": 1,
    "stream": False,
}
response = openai.ChatCompletion.create(
    "gpt-4-32k",
    **request_data,
)
print(json.dumps(response, indent=4))

### The response of Compressed Prompt (Correct in 10x Compression)

In [ ]:
# Setup LLMLingua
from llmlingua import PromptCompressor

llm_lingua = PromptCompressor()

In [ ]:
contexts_list = contexts.split("\n")
contexts_list = [
    "\n".join(contexts_list[ii : ii + 4]) for ii in range(0, len(contexts_list), 4)
]

In [ ]:
# 8x Compression
compressed_prompt = llm_lingua.compress_prompt(
    contexts_list,
    instruction=instruction,
    question=question,
    target_token=2000,
    condition_compare=True,
    condition_in_question="after",
    rank_method="longllmlingua",
    use_sentence_level_filter=False,
    context_budget="+100",
    dynamic_context_compression_ratio=0.4,  # enable dynamic_context_compression_ratio
    reorder_context="sort",
)
message = [
    {"role": "user", "content": compressed_prompt["compressed_prompt"]},
]

request_data = {
    "messages": message,
    "max_tokens": 100,
    "temperature": 0,
    "top_p": 1,
    "n": 1,
    "stream": False,
}
response = openai.ChatCompletion.create(
    "gpt-4-32k",
    **request_data,
)

print(json.dumps(compressed_prompt, indent=4))
print("Response:", response)